# Gemma 4 Explorer

This notebook is for getting familiar with a real Gemma 4 checkpoint while keeping our own learning code nearby.

The path here is:

1. Set up the repo and Hugging Face dependencies.
2. Load the smallest Gemma 4 instruction model by default: `google/gemma-4-E2B-it`.
3. Wrap the model with our own tiny `GemmaChatModel` interface.
4. Ask it a prompt.
5. Inspect the config and weight shapes: vocabulary, embeddings, attention heads, Q/K/V projections, and parameter placement.

This is not a full from-scratch Gemma implementation yet. It is the bridge step: use the real model as a reference object, then gradually rebuild pieces under it as we learn what the shapes mean.

## Setup

Run this first. Locally, it finds the repo root. In Colab, it clones this branch and installs the dependencies needed for Transformers-based Gemma loading.

For local AMD GPU use, start with `USE_4BIT = False`. PyTorch ROCm on Windows reports the AMD GPU through `torch.cuda`, but `bitsandbytes` 4-bit loading is usually a NVIDIA/CUDA path. If you want a quantized chat-only path later, Ollama/LM Studio GGUF is a better local AMD route; this notebook focuses on Hugging Face weights because we want to inspect tensors.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/markschatza/llm-from-scratch.git"
REPO_REF = "codex/gemma-4-explorer"
REPO_DIR = Path("/content/llm-from-scratch")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pretrain" / "gemma_explorer.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root. Run this notebook from inside the llm-from-scratch checkout."
    )

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch",
            "transformers",
            "accelerate",
            "safetensors",
            "huggingface-hub",
            "sentencepiece",
        ],
        check=True,
    )
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
    PROJECT_ROOT = REPO_DIR
else:
    PROJECT_ROOT = find_project_root(Path.cwd())

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"project root: {PROJECT_ROOT}")
if IN_COLAB:
    ref = subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "branch", "--show-current"], text=True).strip()
    print(f"active branch: {ref}")

## Imports And Device Check

On the AMD Windows build, a working GPU usually appears as `cuda` from PyTorch even though the backend is HIP/ROCm. The important check is whether tensors can move to `cuda` and whether `torch.version.hip` is populated.

In [ ]:
import torch

from pretrain.gemma_explorer import (
    DEFAULT_MODEL_ID,
    GenerationSettings,
    attention_projection_shapes,
    choose_dtype,
    embedding_shapes,
    load_gemma_chat_model,
    parameter_summary,
    summarize_config,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("hip version:", getattr(torch.version, "hip", None))
if torch.cuda.is_available():
    print("device 0:", torch.cuda.get_device_name(0))
print("chosen dtype:", choose_dtype())

## Model Choice

The default is the smallest Gemma 4 instruction checkpoint. You can change `MODEL_ID` before loading.

Useful starting points:

- `google/gemma-4-E2B-it`: smallest instruction model, best first local GPU target.
- `google/gemma-4-E4B-it`: larger small model; use after E2B works.
- `google/gemma-4-E2B`: base model without instruction tuning; better for raw next-token behavior, less pleasant for chat.

You may need to accept the model terms on Hugging Face and run `huggingface-cli login` locally, or `from huggingface_hub import notebook_login; notebook_login()` in Colab.

In [ ]:
MODEL_ID = DEFAULT_MODEL_ID
USE_4BIT = False  # Keep False for AMD Windows ROCm. Set True only if your stack supports bitsandbytes.
MAX_NEW_TOKENS = 128

MODEL_ID

## Inspect Config Before Loading Weights

This downloads only the small `config.json`, not the model tensors. It is the quickest way to see the architecture numbers before using VRAM.

In [ ]:
from transformers import AutoConfig

hf_config = AutoConfig.from_pretrained(MODEL_ID)
summarize_config(hf_config)

## Load The Weights

This is the first heavyweight cell. It downloads model files into the Hugging Face cache and places the model on the available device. Expect the first run to take a while.

In [ ]:
chat = load_gemma_chat_model(
    MODEL_ID,
    load_in_4bit=USE_4BIT,
    device_map="auto",
    attn_implementation="sdpa",
)

print("loaded:", MODEL_ID)
print("model device:", chat.device)
print("model dtype:", chat.dtype)

## Talk To The Model

This uses our wrapper class, not the raw Transformers calls. That gives us a stable place to add our own sampling, logging, or eventually a partial model implementation underneath.

In [ ]:
answer = chat.reply(
    "Explain Q, K, and V in attention in one concrete analogy, then give the tensor-shape version.",
    system_prompt="You are a concise tutor for someone building an LLM from scratch.",
    settings=GenerationSettings(max_new_tokens=MAX_NEW_TOKENS, temperature=0.7, top_p=0.95),
)
print(answer)

## Config: The Architecture Numbers

These are the numbers to line up against our own tiny Transformer:

- `vocab_size`: how many token IDs the tokenizer/model knows.
- `hidden_size`: embedding width; our tiny model called this `n_embd`.
- `num_hidden_layers`: how many Transformer blocks are stacked.
- `num_attention_heads`: query heads.
- `num_key_value_heads`: key/value heads. If this is smaller than query heads, the model uses grouped-query attention.
- `head_dim`: width of one attention head.
- `intermediate_size`: MLP expansion width inside each block.

In [ ]:
config_summary = summarize_config(chat.model.config)
config_summary

## Parameter Count And Placement

This answers: how many parameters did we load, what dtype are they in, and where did they land?

In [ ]:
parameter_summary(chat.model)

## Embedding And Vocabulary Weights

The token embedding table is usually shaped like `(vocab_size, hidden_size)`. If the LM head is tied to the input embeddings, it may share storage with that table; if not, it is another large matrix shaped like `(vocab_size, hidden_size)`.

In [ ]:
embedding_shapes(chat.model)

## Attention Projection Shapes

In our tiny model, one `qkv` linear layer produced Q, K, and V together. Many production models store them as separate projections:

- `q_proj`: hidden state to queries.
- `k_proj`: hidden state to keys.
- `v_proj`: hidden state to values.
- `o_proj`: attention output back to hidden size.

For grouped-query attention, K and V can be narrower than Q because fewer key/value heads are shared across more query heads.

In [ ]:
attention_projection_shapes(chat.model, limit=32)

## Inspect A Single Layer More Closely

This finds layer-0 attention weights and prints the shapes together. The exact parameter names may change across model classes, but the shape pattern is what matters.

In [ ]:
for row in attention_projection_shapes(chat.model, limit=128):
    if ".0." in row["name"] or "layers.0" in row["name"]:
        print(f"{row['name']}: shape={row['shape']} dtype={row['dtype']} device={row['device']}")

## Map Gemma Back To Our Tiny Transformer

Use this cell as a translation table. It does not copy weights. It just turns the loaded Gemma config into the closest equivalent field names from `TransformerConfig`.

In [ ]:
from pretrain.transformer import TransformerConfig

cfg = summarize_config(chat.model.config)
rough_tiny_config_names = TransformerConfig(
    vocab_size=cfg["vocab_size"],
    context_length=min(cfg["max_position_embeddings"] or 2048, 2048),
    n_embd=cfg["hidden_size"],
    n_head=cfg["num_attention_heads"],
    n_layer=cfg["num_hidden_layers"],
)
rough_tiny_config_names

## Small Weight Slice

Looking at raw weights is mostly gibberish, but it is useful to see that they are just tensors. This pulls a tiny slice from the first embedding-like matrix so the numbers are visible without dumping anything huge.

In [ ]:
rows = embedding_shapes(chat.model)
first_embedding_name = rows[0]["name"]
params = dict(chat.model.named_parameters())
small_slice = params[first_embedding_name][:3, :8].detach().float().cpu()
print(first_embedding_name)
print(small_slice)

## Cleanup

Run this if you want to release GPU memory before trying a different checkpoint in the same kernel.

In [ ]:
# import gc
# del chat
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()